# 65. Gradio 인터페이스 — ToxGuard 데모 앱분자 SMILES를 입력하면 독성구조 진단, 규칙기반 치환, 관련 선례를바로 확인할 수 있는 웹 인터페이스. `share=True`로 실행하면 Colab세션이 켜져 있는 동안 유효한 공개 링크가 생성됩니다.

In [ ]:
!pip install rdkit -q!pip install gradio -q!pip install openai -q

In [ ]:
from google.colab import userdatatoken = userdata.get('GITHUB_TOKEN')!git clone https://{token}@github.com/Dec32th/laidd-2026.git%cd /content/laidd-2026!pwd

In [ ]:
!git config --global user.email "hyekyeong.w@gmail.com"!git config --global user.name "Dec32th"

In [ ]:
import syssys.path.append('.')from rdkit import Chemfrom PIL import Imageimport iofrom src.tools.toxicophore_detector import detect_toxicophoresfrom src.tools.replacement_library import get_replacement_candidatesfrom src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memoryfrom src.tools.precedent_library import get_precedentsfrom src.tools.visualization import visualize_fix_processprint("✅ 핵심 모듈 로드 완료")

In [ ]:
def analyze_molecule(smiles):    mol = Chem.MolFromSmiles(smiles)    if mol is None:        return "❌ 유효하지 않은 SMILES입니다.", None, ""    problems = detect_toxicophores(smiles)    if not problems:        return "✅ 진단된 독성 구조가 없습니다.", None, ""    diag_text = f"진단된 문제 {len(problems)}건:\n"    for p in problems:        diag_text += f"  - {p['rule_name']}\n"    clear_failure_memory()    result = iterative_fix_loop(smiles, max_iterations=10, candidate_idx=0)    diag_text += f"\n최종 상태: {result['status']}\n"    diag_text += f"최종 분자: {result.get('final_smiles', '(없음)')}\n"    precedent_text = ""    seen_rules = set()    for h in result['history']:        rule = h.get('fixed_rule')        if rule and rule not in seen_rules:            seen_rules.add(rule)            prec = get_precedents(rule)            if prec:                precedent_text += f"[{rule}]\n{prec}\n\n"    if not precedent_text:        precedent_text = "(등록된 선례 없음)"    img = None    if len(result['history']) > 0:        viz = visualize_fix_process(result, mols_per_row=2, sub_img_size=(800, 800))        raw_img = viz['image_2d']        if hasattr(raw_img, 'data'):            img = Image.open(io.BytesIO(raw_img.data))        else:            img = raw_img    return diag_text, img, precedent_text

함수 자체를 먼저 검증합니다.

In [ ]:
diag, img, prec = analyze_molecule("CCCCCCCCCCCCCCCCOCCO")print(diag)print("---")print(prec)img

## Gradio 인터페이스 실행

In [ ]:
import gradio as grdemo = gr.Interface(    fn=analyze_molecule,    inputs=gr.Textbox(        label="분자 SMILES",        placeholder="예: CCCCCCCCCCCCCCCCOCCO",        value="CCCCCCCCCCCCCCCCOCCO",    ),    outputs=[        gr.Textbox(label="진단 및 치환 결과"),        gr.Image(label="치환 과정 시각화"),        gr.Textbox(label="관련 선례 (ChEMBL/GtoPdb 근거)"),    ],    title="ToxGuard — 분자 독성구조 진단 및 치환 데모",    description=(        "SMILES를 입력하면 BRENK/PAINS 규칙 기반으로 독성구조를 진단하고, "        "화학적 치환 라이브러리 기반으로 개선된 분자를 제안합니다. "        "JUMP AI 2026 공모전(팀명 MorForge) 제출작 기반, 지속 개발 중인 프로젝트입니다."    ),    examples=[        ["CCCCCCCCCCCCCCCCOCCO"],        ["CN=C=O"],        ["Oc1ccc(cc1)C=O"],    ],    flagging_mode="never",)demo.launch(share=True, debug=True)